In [1]:
import os
import cv2
import torch
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from torchvision import models, transforms
import torch.nn as nn

# =========================
# CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
IMG_DIR  = "/kaggle/input/rsna-bone-age/boneage-training-dataset/boneage-training-dataset"
MODEL_PATH = "/kaggle/input/baseline-model-6-5-mae/pytorch/default/1/stage1_localization_best.pt"

OUT = "./outputs"

DIRS = {
    "hand_hm": f"{OUT}/heatmap_hand",
    "r1_hm":   f"{OUT}/heatmap_r1",
    "r2_hm":   f"{OUT}/heatmap_r2",
    "hand":    f"{OUT}/hand",
    "r1":      f"{OUT}/region1",
    "r2":      f"{OUT}/region2",
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

IMG_SIZE = 299
THRESH = 40

# =========================
# MODEL
# =========================
class Stage1LocalizationNet(nn.Module):
    def __init__(self, num_bins=240):
        super().__init__()

        base = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True
        )

        self.features = nn.Sequential(
            base.Conv2d_1a_3x3, base.Conv2d_2a_3x3, base.Conv2d_2b_3x3,
            base.maxpool1, base.Conv2d_3b_1x1, base.Conv2d_4a_3x3,
            base.maxpool2, base.Mixed_5b, base.Mixed_5c, base.Mixed_5d,
            base.Mixed_6a, base.Mixed_6b, base.Mixed_6c, base.Mixed_6d,
            base.Mixed_6e, base.Mixed_7a, base.Mixed_7b, base.Mixed_7c,
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gender_fc = nn.Linear(1, 16)

        self.head = nn.Sequential(
            nn.Linear(2048 + 16, 1024),
            nn.ReLU(),
            nn.Linear(1024, num_bins)
        )

    def forward(self, x, gender):
        fmap = self.features(x)
        pooled = self.pool(fmap).flatten(1)
        g = self.gender_fc(gender.unsqueeze(1))
        fused = torch.cat([pooled, g], dim=1)
        out = self.head(fused)
        return out, fmap

# =========================
# TRANSFORMS
# =========================
tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# =========================
# UTILS (author-faithful + safe)
# =========================
def find_max_component(mask):
    contours,_ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        return mask

    areas = [cv2.contourArea(c) for c in contours]
    max_id = np.argmax(areas)

    for i,c in enumerate(contours):
        if i != max_id:
            cv2.fillConvexPoly(mask, c, 0)

    return mask


def crop(img, mask):
    idx = np.where(mask > 0)
    if len(idx[0]) == 0:
        return None

    top = np.min(idx[0])
    bottom = np.max(idx[0])
    left = np.min(idx[1])
    right = np.max(idx[1])

    return img[top:bottom, left:right]


def generate_heatmap(model, img_bgr, gender):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inp = tf(rgb).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _, fmap = model(inp, gender)

    weights = model.head[-1].weight.mean(dim=0)

    cam = torch.zeros(fmap.shape[2:], device=DEVICE)
    for i, w in enumerate(weights):
        cam += w * fmap[0, i]

    # ✅ FIX HERE
    cam = cam.detach().cpu().numpy()

    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)
    cam = cv2.resize(cam, (img_bgr.shape[1], img_bgr.shape[0]))
    heatmap = np.uint8(255 * cam)

    return heatmap



def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in os.listdir(folder_path):
            full = os.path.join(folder_path, f)
            if os.path.isfile(full):
                z.write(full, arcname=f)

# =========================
# MAIN PIPELINE
# =========================
def main():
    print("Loading model...")
    model = Stage1LocalizationNet().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE), strict=False)
    model.eval()

    df = pd.read_csv(CSV_PATH)
    valid_ids = set(int(f[:-4]) for f in os.listdir(IMG_DIR) if f.endswith(".png"))
    df = df[df["id"].isin(valid_ids)]

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_id = int(row["id"])
        gender = torch.tensor([float(row["male"])], device=DEVICE)

        img_path = f"{IMG_DIR}/{img_id}.png"
        full_img = cv2.imread(img_path)
        if full_img is None:
            continue

        # ===== Stage 1: Hand =====
        hm_hand = generate_heatmap(model, full_img, gender)
        cv2.imwrite(f"{DIRS['hand_hm']}/{img_id}.jpg", hm_hand)

        _, mask = cv2.threshold(hm_hand, THRESH, 255, cv2.THRESH_BINARY)
        mask = find_max_component(mask)

        hand = crop(full_img, mask)
        if hand is None:
            continue

        cv2.imwrite(f"{DIRS['hand']}/{img_id}.jpg", hand)

        # ===== Stage 2: Region 1 =====
        hm_r1 = generate_heatmap(model, hand, gender)
        cv2.imwrite(f"{DIRS['r1_hm']}/{img_id}.jpg", hm_r1)

        _, mask = cv2.threshold(hm_r1, THRESH, 255, cv2.THRESH_BINARY)
        mask = find_max_component(mask)

        r1 = crop(hand, mask)
        if r1 is None:
            continue

        cv2.imwrite(f"{DIRS['r1']}/{img_id}.jpg", r1)

        # ===== Stage 3: Region 2 =====
        hm_r2 = generate_heatmap(model, r1, gender)
        cv2.imwrite(f"{DIRS['r2_hm']}/{img_id}.jpg", hm_r2)

        _, mask = cv2.threshold(hm_r2, THRESH, 255, cv2.THRESH_BINARY)
        mask = find_max_component(mask)

        r2 = crop(r1, mask)
        if r2 is None:
            continue

        cv2.imwrite(f"{DIRS['r2']}/{img_id}.jpg", r2)

    print("Zipping patch folders...")

    zip_folder(DIRS["hand"], f"{OUT}/hand.zip")
    zip_folder(DIRS["r1"],   f"{OUT}/region1.zip")
    zip_folder(DIRS["r2"],   f"{OUT}/region2.zip")

    print("Done.")

if __name__ == "__main__":
    main()


Loading model...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 243MB/s] 
100%|██████████| 12611/12611 [50:44<00:00,  4.14it/s]


Zipping patch folders...
Done.
